In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

─────────────────────────────────────────────────────────────<br>
 CONFIG<br>
─────────────────────────────────────────────────────────────

In [ ]:
DATASET_DIR           = 'Dataset'
DROP_COLUMNS          = ['time', 'id', 'unnamed: 0', 'index']
LABEL_COLUMN_HINTS    = ['class', 'label', 'target', 'outcome', 'diagnosis',
                         'result', 'category', 'fraud', 'y', 'output']
MAX_SAMPLES_PER_CLASS = 2500

─────────────────────────────────────────────────────────────<br>
 HYPERPARAMETER SEARCH SPACES<br>
<br>
 Each model has named params with discrete option lists.<br>
 The GA encodes each param as an integer index into its list.<br>
<br>
 KNN : n_neighbors, weights, metric         → 3 param genes<br>
 SVM : C, gamma, kernel                     → 3 param genes<br>
 RF  : n_estimators, max_depth, min_samples → 3 param genes<br>
─────────────────────────────────────────────────────────────

In [ ]:
PARAM_SPACE = {
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11, 15],
        'weights':     ['uniform', 'distance'],
        'metric':      ['euclidean', 'manhattan'],
    },
    'SVM': {
        'C':      [0.1, 0.5, 1.0, 5.0, 10.0],
        'gamma':  ['scale', 'auto'],
        'kernel': ['rbf', 'linear', 'poly'],
    },
    'RF': {
        'n_estimators':      [50, 100, 200],
        'max_depth':         [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
    },
}

In [ ]:
def decode_params(model_name, param_genes):
    """Integer gene array → concrete hyperparameter dict."""
    params = {}
    for gene, (pname, options) in zip(param_genes, PARAM_SPACE[model_name].items()):
        params[pname] = options[int(gene) % len(options)]
    return params

In [ ]:
def build_model(model_name, param_genes):
    """Instantiate a model from its name + param genes."""
    p = decode_params(model_name, param_genes)
    if model_name == 'KNN':
        return KNeighborsClassifier(**p)
    elif model_name == 'SVM':
        return SVC(random_state=42, max_iter=2000, **p)
    elif model_name == 'RF':
        return RandomForestClassifier(random_state=42, **p)

In [ ]:
def default_model(model_name):
    """Return a model with sklearn defaults for baseline comparison."""
    if model_name == 'KNN':
        return KNeighborsClassifier()
    elif model_name == 'SVM':
        return SVC(random_state=42, max_iter=2000)
    elif model_name == 'RF':
        return RandomForestClassifier(n_estimators=100, random_state=42)

─────────────────────────────────────────────────────────────<br>
 COMBINED CHROMOSOME<br>
<br>
 Layout:  [ f0 f1 ... fn | p0 p1 p2 ]<br>
           <-- binary -->   <-- ints --><br>
<br>
 Feature bits  (binary 0/1)  : length = n_features<br>
 Param genes   (int indices) : length = number of hyperparams (3)<br>
─────────────────────────────────────────────────────────────

In [ ]:
def random_chromosome(n_features, model_name):
    feat_bits   = np.random.randint(0, 2, n_features)
    param_genes = np.array([
        np.random.randint(0, len(opts))
        for opts in PARAM_SPACE[model_name].values()
    ])
    return np.concatenate([feat_bits, param_genes])

In [ ]:
def split_chromosome(chromo, n_features):
    return chromo[:n_features].astype(int), chromo[n_features:].astype(int)

─────────────────────────────────────────────────────────────<br>
 GA OPERATORS<br>
─────────────────────────────────────────────────────────────

In [ ]:
def fitness(chromo, model_name, X, y, n_features):
    """
    Joint fitness over features AND hyperparameters.
    Prioritises accuracy heavily, using feature reduction as a tie-breaker.
    fitness = 0.99 x CV_accuracy + 0.01 x feature_reduction_bonus
    """
    feat_bits, param_genes = split_chromosome(chromo, n_features)
    if not any(feat_bits):
        return 0.0
    model  = build_model(model_name, param_genes)
    score  = cross_val_score(model, X[:, feat_bits == 1], y, cv=5).mean()
    bonus  = 1.0 - feat_bits.sum() / n_features
    return 0.99 * score + 0.01 * bonus

In [ ]:
def crossover(p1, p2):
    """
    Two-point crossover on the full chromosome.
    Operates uniformly across feature bits AND param genes.
    """
    n = len(p1)
    pt1, pt2 = sorted(np.random.choice(range(1, n), 2, replace=False))
    c1 = np.concatenate([p1[:pt1], p2[pt1:pt2], p1[pt2:]])
    c2 = np.concatenate([p2[:pt1], p1[pt1:pt2], p2[pt2:]])
    return c1, c2

In [ ]:
def mutate(chromo, n_features, model_name, rate=0.05):
    """
    Type-aware mutation:
      Feature bits  -> bit-flip
      Param genes   -> resample uniformly from that param's valid range
    """
    chromo = chromo.copy().astype(int)
    for i in range(n_features):
        if np.random.rand() < rate:
            chromo[i] ^= 1
    for j, opts in enumerate(PARAM_SPACE[model_name].values()):
        if np.random.rand() < rate:
            chromo[n_features + j] = np.random.randint(0, len(opts))
    return chromo

In [ ]:
def run_ga(model_name, X, y, popu=20, gener=10):
    """
    Genetic Algorithm that jointly optimises:
      - Which features to use   (binary chromosome section)
      - Which hyperparameters   (integer chromosome section)
    Returns: (best_chromosome, fitness_history_list)
    """
    n_features = X.shape[1]
    pop        = [random_chromosome(n_features, model_name) for _ in range(popu)]
    best_chromo, best_fit = None, -1.0
    history = []
    for gen in range(gener):
        scores = [fitness(c, model_name, X, y, n_features) for c in pop]
        order  = np.argsort(scores)[::-1]
        pop    = [pop[i] for i in order]
        gen_best = scores[order[0]]
        history.append(gen_best)
        if gen_best > best_fit:
            best_fit    = gen_best
            best_chromo = pop[0].copy()

        # Elitism: top 2 pass unchanged
        next_pop = pop[:2]
        while len(next_pop) < popu:
            i1, i2 = np.random.choice(max(2, popu // 2), 2, replace=False)
            c1, c2 = crossover(pop[i1], pop[i2])
            next_pop += [mutate(c1, n_features, model_name),
                         mutate(c2, n_features, model_name)]
        pop = next_pop[:popu]
        feat_bits, param_genes = split_chromosome(pop[0], n_features)
        params = decode_params(model_name, param_genes)
        print(f"  [{model_name}] Gen {gen+1:02d}/{gener} | "
              f"Fitness={gen_best:.4f} | "
              f"Features={int(feat_bits.sum())}/{n_features} | "
              f"Params={params}")
    return best_chromo, history

─────────────────────────────────────────────────────────────<br>
 DATA LOADING<br>
─────────────────────────────────────────────────────────────

In [ ]:
def _encode_labels(series):
    # Use Pandas' native type checker to avoid crashes with StringDtype
    if not pd.api.types.is_numeric_dtype(series):
        le = LabelEncoder()
        y  = le.fit_transform(series.astype(str))
    else:
        y = series.values.astype(int)
    return y if len(np.unique(y)) >= 2 else None

In [ ]:
def stratified_sample(X, y, max_per_class=MAX_SAMPLES_PER_CLASS):
    indices = []
    for cls in np.unique(y):
        idx     = np.where(y == cls)[0]
        sampled = np.random.choice(idx, size=min(len(idx), max_per_class), replace=False)
        indices.append(sampled)
    idx = np.concatenate(indices)
    np.random.shuffle(idx)
    return X[idx], y[idx]

In [ ]:
def try_load_file(filepath):
    filename = os.path.basename(filepath)

    # 1. Swapped ' ' for r'\s+' to handle multiple spaces natively
    for sep in [',', r'\s+', '\t', ';']:
        try:
            # 2. Explicit header=None for the peek to prevent mangling first-row data
            peek = pd.read_csv(filepath, sep=sep, engine='python',
                               nrows=5, on_bad_lines='skip', header=None)

            # If it couldn't split into multiple columns, try next separator
            if peek.shape[1] < 2:
                continue

            # Heuristic: Check if the first row acts as a string header
            first_row = peek.iloc[0].astype(str)
            has_header = any(not c.replace('.', '', 1).lstrip('-').isdigit() for c in first_row)
            df = pd.read_csv(filepath, sep=sep,
                             header=0 if has_header else None,
                             engine='python', on_bad_lines='skip')

            # 3. FIX: Drop empty columns (handles trailing commas generating NaN columns)
            df = df.dropna(axis=1, how='all')
            if df.shape[1] < 2 or df.shape[0] < 10:
                continue

            # Clean header names if present
            if has_header:
                clm = {str(c).lower().strip(): c for c in df.columns}
                to_drop = [clm[k] for k in DROP_COLUMNS if k in clm]
                if to_drop:
                    df = df.drop(columns=to_drop)
            y = None

            # Try finding target by hint
            if has_header:
                clm = {str(c).lower().strip(): c for c in df.columns}
                for hint in LABEL_COLUMN_HINTS:
                    if hint in clm:
                        y = _encode_labels(df[clm[hint]])
                        if y is not None:
                            print(f"  [INFO]  {filename}: label='{clm[hint]}'")
                            df = df.drop(columns=[clm[hint]])
                            break

            # Fallback: use the last column as target
            if y is None:
                y = _encode_labels(df.iloc[:, -1])
                if y is not None:
                    df = df.iloc[:, :-1]
                else:
                    continue

            # Convert remaining features to numeric, drop irredeemable ones
            df = df.apply(pd.to_numeric, errors='coerce')
            df = df.dropna(axis=1, thresh=int(0.8 * len(df)))
            df = df.fillna(df.median(numeric_only=True))
            if df.shape[1] < 2:
                continue
            X = df.values.astype(float)
            classes, counts = np.unique(y, return_counts=True)
            total = len(y)
            X, y = stratified_sample(X, y)

            # 4. FIX: Ensure no zero-variance NaNs trip up the scaler
            X = np.nan_to_num(X)
            X = StandardScaler().fit_transform(X)
            if total > len(y):
                print(f"  [INFO]  {filename}: sampled {len(y)} from {total} rows "
                      f"(dist: {dict(zip(classes.tolist(), counts.tolist()))})")
            return X, y
        except Exception as e:
            import traceback
            # 5. DEBUGGING: Uncomment the print statement below to see the exact error
            print(f"Error on {filename} with sep '{sep}': {e}\n{traceback.format_exc()}")
            continue
    print(f"  [SKIP]  {filename}: could not parse")
    return None

In [ ]:
def get_datasets():
    datasets = {}
    bc = load_breast_cancer()
    datasets['Breast Cancer'] = (StandardScaler().fit_transform(bc.data), bc.target)
    if not os.path.isdir(DATASET_DIR):
        print(f"[WARN] '{DATASET_DIR}/' not found.")
        return datasets
    supported  = ('.csv', '.data', '.txt', '.tsv')
    skip_words = ['names', 'readme', 'index', 'info', 'description', 'license']
    print(f"\n── Scanning '{DATASET_DIR}/' ──────────────────────────────")
    for root, dirs, files in os.walk(DATASET_DIR):
        for filename in sorted(files):
            if os.path.splitext(filename)[1].lower() not in supported:
                continue
            if any(kw in filename.lower() for kw in skip_words):
                continue
            filepath = os.path.join(root, filename)
            result = try_load_file(filepath)
            if result is not None:
                X, y = result
                name = (os.path.splitext(filename)[0]
                        .replace('_', ' ').replace('-', ' ').title())
                datasets[name] = (X, y)
                print(f"  [OK]    {name:30s}  shape={X.shape}  classes={np.unique(y)}")
    print("─" * 55 + "\n")
    return datasets

─────────────────────────────────────────────────────────────<br>
 EVALUATION + PLOTTING<br>
─────────────────────────────────────────────────────────────

In [ ]:
def evaluate_models():
    datasets    = get_datasets()
    model_names = ['KNN', 'SVM', 'RF']
    results     = []
    ga_histories = {}
    for ds_name, (X, y) in datasets.items():
        print(f"\n{'='*65}")
        print(f"  {ds_name}  |  shape={X.shape}  |  classes={np.unique(y)}")
        print(f"{'='*65}")
        n_feat = X.shape[1]
        for m_name in model_names:
            print(f"\n  [{m_name}]")

            # Baseline: default hyperparams, ALL features
            raw = cross_val_score(default_model(m_name), X, y, cv=5).mean()
            print(f"    Baseline (default params, all {n_feat} features): {raw:.4f}")

            # GA: jointly optimise features + hyperparams
            best_chromo, history = run_ga(m_name, X, y, popu=20, gener=10)
            ga_histories[f'{ds_name}|{m_name}'] = history
            feat_bits, param_genes = split_chromosome(best_chromo, n_feat)
            best_params = decode_params(m_name, param_genes)
            n_sel       = int(feat_bits.sum())
            ga_acc      = (cross_val_score(build_model(m_name, param_genes),
                                           X[:, feat_bits == 1], y, cv=5).mean()
                           if n_sel > 0 else raw)
            delta = ga_acc - raw
            print(f"    GA result ({n_sel}/{n_feat} features | params={best_params}): "
                  f"{ga_acc:.4f}  [{'+' if delta >= 0 else ''}{delta:.4f}]")
            results.append(dict(
                dataset     = ds_name,
                model       = m_name,
                raw         = raw,
                ga          = ga_acc,
                n_total     = n_feat,
                n_selected  = n_sel,
                best_params = best_params,
            ))

    # ── Plot 1: Accuracy comparison bar chart ─────────────────
    labels   = [f"{r['dataset']}\n{r['model']}" for r in results]
    raw_vals = [r['raw'] for r in results]
    ga_vals  = [r['ga']  for r in results]
    x        = np.arange(len(labels))
    w        = 0.35
    n_ds     = len(datasets)
    n_mod    = len(model_names)
    fig, ax = plt.subplots(figsize=(max(12, len(labels) * 1.4), 7))
    b1 = ax.bar(x - w/2, raw_vals, w,
                label='Baseline (default params, all features)',
                color='#4C72B0', alpha=0.85)
    b2 = ax.bar(x + w/2, ga_vals, w,
                label='GA Optimised (features + hyperparams)',
                color='#DD8452', alpha=0.85)
    for rect in list(b1) + list(b2):
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.003,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)
    for rect, r in zip(b2, results):
        dropped = r['n_total'] - r['n_selected']
        if dropped > 0:
            ax.text(rect.get_x() + rect.get_width()/2,
                    rect.get_height() + 0.027,
                    f'↓{dropped}f', ha='center', fontsize=7,
                    color='green', style='italic')
    for d in range(1, n_ds):
        ax.axvline(x=d * n_mod - 0.5, color='gray',
                   linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_ylabel('5-Fold Cross-Validation Accuracy', fontsize=11)
    ax.set_title('GA Optimisation — Baseline vs GA (Features + Hyperparameters)',
                 fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.legend(fontsize=9)
    ax.set_ylim([max(0, min(raw_vals + ga_vals) - 0.08), 1.08])
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('comparison_plot.png', dpi=150)
    print("\n✅  Accuracy chart saved → comparison_plot.png")
    plt.show()

    # ── Plot 2: Fitness curves ────────────────────────────────
    n_keys = len(ga_histories)
    cols   = min(3, n_keys)
    rows   = (n_keys + cols - 1) // cols
    fig2, axes = plt.subplots(rows, cols,
                              figsize=(cols * 5, rows * 3.5),
                              squeeze=False)
    fig2.suptitle('GA Fitness Over Generations (Features + Hyperparameters)',
                  fontsize=12, fontweight='bold')
    for idx, (key, hist) in enumerate(ga_histories.items()):
        r, c   = divmod(idx, cols)
        ds, mn = key.split('|')
        ax2 = axes[r][c]
        ax2.plot(range(1, len(hist)+1), hist, marker='o',
                 markersize=4, color='#DD8452', linewidth=2)
        ax2.set_title(f'{ds} — {mn}', fontsize=9, fontweight='bold')
        ax2.set_xlabel('Generation', fontsize=8)
        ax2.set_ylabel('Best Fitness', fontsize=8)
        ax2.set_ylim([max(0, min(hist) - 0.02), 1.0])
        ax2.grid(alpha=0.3)
        ax2.tick_params(labelsize=7)
    for idx in range(n_keys, rows * cols):
        r, c = divmod(idx, cols)
        axes[r][c].set_visible(False)
    plt.tight_layout()
    plt.savefig('fitness_curves.png', dpi=150)
    print("✅  Fitness curves saved → fitness_curves.png")
    plt.show()

    # ── Summary table ─────────────────────────────────────────
    print("\n" + "="*95)
    print(f"{'Dataset':<20} {'Model':<5} {'Baseline':>9} {'GA Acc':>7} "
          f"{'Delta':>7}  {'Feats':>7}  Best Hyperparameters")
    print("="*95)
    for r in results:
        d = r['ga'] - r['raw']
        print(f"{r['dataset']:<20} {r['model']:<5} "
              f"{r['raw']:>9.4f} {r['ga']:>7.4f} "
              f"{'+' if d>=0 else ''}{d:>6.4f}  "
              f"{r['n_selected']:>3}/{r['n_total']:<3}  "
              f"{r['best_params']}")
    print("="*95)

In [ ]:
if __name__ == "__main__":
    evaluate_models()